# Reachy grasping: see an object with the head cameras, then pick it up
Builds on `reachy_ik_test.ipynb`. Pipeline:

1. **Stereo calibration** (once per zoom setting): checkerboard → intrinsics of both head cameras + the left→right transform.
2. **Teach** a grasp hand orientation and a fixed head pose.
3. **Hand–eye calibration**: the arm moves an ArUco marker stuck on the gripper through a grid of points; the marker is
   triangulated in both cameras and matched to the arm's forward kinematics → camera→torso transform. No head geometry needed,
   but **the head must be in the same pose for calibration and grasping** (the notebook stores and restores it).
4. **Locate**: OWL-ViT finds the object by text in both images, MobileSAM segments it, mask centroids are triangulated → xyz in the torso frame.
5. **Grasp**: plan every waypoint with IK first (abort if any is off by > 1 cm), then pre-grasp → grasp → close → lift.

Frame: torso, **x forward, y left, z up**, metres. Files written next to this notebook: `stereo_calib.npz`, `grasp_setup.npz`, `handeye.npz`, `grasp_debug.jpg`.

**You need:** the calib.io checkerboard (6×5 squares of 100 mm), and a printed ArUco marker
(4x4_50, id 0, ~3 cm) taped flat on the gripper where both head cameras can see it. Needs OpenCV ≥ 4.7 (or `opencv-contrib-python` for older) and `pollen-vision`.
No GUI windows are used: everything prints or saves images, so it works over SSH.

In [ ]:
import time
import numpy as np, cv2
from reachy_sdk import ReachySDK
from reachy_sdk.camera import ZoomLevel
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode

np.set_printoptions(precision=3, suppress=True)
reachy = ReachySDK(host='localhost')

ARM_NAME = 'l_arm'
ARM = reachy.l_arm
GRIPPER = reachy.l_arm.l_gripper
CHAIN = list(ARM.kinematics_chain.values())        # 7 arm joints
NECK = list(reachy.head.kinematics_chain.values())   # neck_roll, neck_pitch, neck_yaw
CAMS = (reachy.left_camera, reachy.right_camera)

ZOOM = ZoomLevel.OUT          # calibration only holds for the zoom it was made at
GRIPPER_OPEN, GRIPPER_CLOSED = 35, -12   # deg, from reachy_movement.ipynb; raise CLOSED for soft objects
MAX_IK_ERR_MM = 10
MAX_JOINT_JUMP = 90           # deg, refuse wild swings between waypoints

STEREO_FILE, SETUP_FILE, HANDEYE_FILE = 'stereo_calib.npz', 'grasp_setup.npz', 'handeye.npz'

for cam in CAMS:
    cam.zoom_level = ZOOM
time.sleep(2)   # let the zoom motors settle
print([j.name for j in CHAIN], [j.name for j in NECK])

In [ ]:
def grab_stereo():
    # Fresh, simultaneous-ish BGR frames from both head cameras.
    for cam in CAMS:
        cam.wait_for_new_frame()
    return CAMS[0].last_frame.copy(), CAMS[1].last_frame.copy()

def stiffen(part_name, joints):
    # Turn a part on without snapping to an old goal: hold where it is now.
    for j in joints:
        j.goal_position = j.present_position
    reachy.turn_on(part_name)

def current_q():
    return [j.present_position for j in CHAIN]

def plan_ik(T_goal, q_seed):
    # IK + accuracy check. Returns (q, err_mm) or (None, reason).
    try:
        q = ARM.inverse_kinematics(T_goal, q0=q_seed)
    except ValueError as e:
        return None, f'IK failed: {e}'
    err_mm = np.linalg.norm(ARM.forward_kinematics(joints_position=q)[:3, 3] - T_goal[:3, 3]) * 1000
    if err_mm > MAX_IK_ERR_MM:
        return None, f'IK only gets within {err_mm:.0f} mm (out of reach with this orientation?)'
    jump = np.abs(np.array(q) - np.array(q_seed)).max()
    if jump > MAX_JOINT_JUMP:
        return None, f'joint swing of {jump:.0f} deg'
    return q, err_mm

def pose(R, xyz):
    T = np.eye(4); T[:3, :3] = R; T[:3, 3] = xyz
    return T

def move_joints(q, duration):
    goto({j: a for j, a in zip(CHAIN, q)}, duration=duration,
         interpolation_mode=InterpolationMode.MINIMUM_JERK)

def set_gripper(angle, duration=1.0):
    goto({GRIPPER: angle}, duration=duration, interpolation_mode=InterpolationMode.MINIMUM_JERK)

### Live view in your browser
The next cell starts a small MJPEG server in the background. It shows both head cameras side by side, plus whatever
the current step is doing: checkerboard corners, the gripper marker, or the detected object.

- **On the robot:** open http://localhost:8090
- **From your laptop:** open `http://<reachy-ip>:8090`, or tunnel it with `ssh -L 8090:localhost:8090 reachy-eth`
  and open http://localhost:8090

The server keeps running while the kernel is alive, and re-running the cell doesn't start a second one.

In [ ]:
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from socketserver import ThreadingMixIn

STREAM_PORT = 8090
STREAM_FPS = 15
_view = {'img': None, 'until': 0.0}
_view_lock = threading.Lock()

def label(img, text, color=(0, 255, 100), row=0):
    cv2.putText(img, text, (10, 25 + 25 * row), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 4)
    cv2.putText(img, text, (10, 25 + 25 * row), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

def publish(img, hold=1.0):
    # Show img in the browser for `hold` seconds, then fall back to the live cameras.
    with _view_lock:
        _view['img'], _view['until'] = img, time.time() + hold

def _current_view():
    with _view_lock:
        if _view['img'] is not None and time.time() < _view['until']:
            return _view['img']
    # Only read last_frame here: wait_for_new_frame() from this thread would steal frames from the main code.
    left, right = CAMS[0].last_frame.copy(), CAMS[1].last_frame.copy()
    return np.hstack([label(left, 'LEFT'), label(right, 'RIGHT', (0, 180, 255))])

class _StreamHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/':
            page = b'<html><body style="margin:0;background:#111"><img src="/stream" style="width:100%"></body></html>'
            self.send_response(200)
            self.send_header('Content-Type', 'text/html')
            self.end_headers()
            self.wfile.write(page)
        elif self.path == '/stream':
            self.send_response(200)
            self.send_header('Content-Type', 'multipart/x-mixed-replace; boundary=frame')
            self.end_headers()
            try:
                while True:
                    ok, jpg = cv2.imencode('.jpg', _current_view(), [cv2.IMWRITE_JPEG_QUALITY, 80])
                    if ok:
                        self.wfile.write(b'--frame\r\nContent-Type: image/jpeg\r\n\r\n' + jpg.tobytes() + b'\r\n')
                    time.sleep(1 / STREAM_FPS)
            except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError):
                pass   # browser tab closed
        else:
            self.send_error(404)

    def log_message(self, *args):
        pass   # keep the notebook output clean

class _StreamServer(ThreadingMixIn, HTTPServer):
    daemon_threads = True
    allow_reuse_address = True

grab_stereo()   # make sure both cameras have a first frame
if '_stream_server' not in globals():
    _stream_server = _StreamServer(('0.0.0.0', STREAM_PORT), _StreamHandler)
    threading.Thread(target=_stream_server.serve_forever, daemon=True).start()
print(f'Live view: http://localhost:{STREAM_PORT}')

## Step 1: Stereo calibration (once; redo if you change `ZOOM`)
Hold the checkerboard in front of the head and move/tilt it slowly. A pair is kept every ~1.5 s when **both** cameras
see the full board, including the white border around the outer squares. The calib.io board is about 50×60 cm, so it
only fits from roughly 1 m or more away. Cover the whole image, including its corners, vary the distance (about 1 to 2 m),
and tilt the board up to about 40° in different directions. Keep it still for a moment at each spot so the frames aren't blurred.

In [ ]:
CHECKERBOARD = (4, 5)    # inner corners: calib.io 6x5-square board -> 5x4 inner corners (orientation doesn't matter)
SQUARE_M = 0.100         # calib.io "Checker Size: 100 mm"; check one square with a ruler
N_PAIRS = 30             # few corners per view, so take a few more views

objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2) * SQUARE_M
crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)

def find_corners(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ok, c = cv2.findChessboardCorners(gray, CHECKERBOARD, cv2.CALIB_CB_ADAPTIVE_THRESH
                                      + cv2.CALIB_CB_NORMALIZE_IMAGE + cv2.CALIB_CB_FAST_CHECK)
    return cv2.cornerSubPix(gray, c, (11, 11), (-1, -1), crit) if ok else None

obj_pts, l_pts, r_pts = [], [], []
last = 0
while len(obj_pts) < N_PAIRS:
    left, right = grab_stereo()
    cl, cr = find_corners(left), find_corners(right)
    both = cl is not None and cr is not None
    kept = both and time.time() - last >= 1.5
    if kept:
        obj_pts.append(objp); l_pts.append(cl); r_pts.append(cr)
        last = time.time()
        print(f'pair {len(obj_pts)}/{N_PAIRS}: move the board')
    # live view: corners drawn, green flash when a pair is kept
    vis = [img.copy() for img in (left, right)]
    for img, c in zip(vis, (cl, cr)):
        if c is not None:
            cv2.drawChessboardCorners(img, CHECKERBOARD, c, True)
    view = np.hstack(vis)
    label(view, f'pairs {len(obj_pts)}/{N_PAIRS}', (0, 255, 0) if kept else (255, 255, 255))
    label(view, 'board seen in both' if both else 'need the whole board in BOTH cameras',
          (0, 255, 0) if both else (0, 0, 255), row=1)
    publish(view, hold=0.5)
IMG_SIZE = left.shape[1::-1]   # (w, h)
print('image size', IMG_SIZE)

In [ ]:
_, K1, D1, _, _ = cv2.calibrateCamera(obj_pts, l_pts, IMG_SIZE, None, None)
_, K2, D2, _, _ = cv2.calibrateCamera(obj_pts, r_pts, IMG_SIZE, None, None)
rms, K1, D1, K2, D2, R_lr, T_lr, _, _ = cv2.stereoCalibrate(
    obj_pts, l_pts, r_pts, K1, D1, K2, D2, IMG_SIZE,
    criteria=crit, flags=cv2.CALIB_FIX_INTRINSIC)
print(f'stereo RMS reprojection error: {rms:.2f} px  (aim for < 1)')
print(f'baseline: {np.linalg.norm(T_lr) * 100:.1f} cm')
np.savez(STEREO_FILE, K1=K1, D1=D1, K2=K2, D2=D2, R=R_lr, T=T_lr, size=IMG_SIZE, zoom=ZOOM.name)

## Step 2: Teach the grasp orientation and the head pose
1. Run the next cell. The head looks down at the table, and the arm goes **compliant**, so hold it.
2. By hand, put the gripper in a front-grasp pose (fingers pointing forward, opening horizontal), somewhere in front of the robot.
3. Run the cell after it to record the hand orientation and head pose, and stiffen the arm.

In [ ]:
LOOK_AT = (0.5, 0.1, -0.35)   # point on the table to stare at (torso frame, m)
reachy.turn_on('head')
reachy.head.look_at(*LOOK_AT, duration=1.5)
reachy.turn_off(ARM_NAME)
print('Arm is compliant: pose the hand, then run the next cell.')

In [ ]:
R_GRASP = ARM.forward_kinematics()[:3, :3]
HEAD_Q = [j.present_position for j in NECK]
stiffen(ARM_NAME, CHAIN + [GRIPPER])
np.savez(SETUP_FILE, R_grasp=R_GRASP, head_q=HEAD_Q, arm_name=ARM_NAME)
print('hand orientation:\n', R_GRASP, '\nhead joints (deg):', np.round(HEAD_Q, 1))

## Step 3: Hand–eye calibration (camera → torso)
The arm visits a grid of points with the taught orientation. At each one, the ArUco marker on the gripper is triangulated
in the camera frame and paired with the arm's forward-kinematics position. A rigid fit (Kabsch) between the two sets gives
the camera→torso transform. Unreachable points are skipped. Adjust the grid if fewer than about 12 points are used,
or if the marker leaves the image.

The next cell (**calibration helpers**) loads the saved calibration files and also writes `aruco_marker.png` for printing.
**In later sessions** run the setup cells, then this helpers cell, then skip to Step 4.

In [ ]:
ARUCO_DICT = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
MARKER_ID = 0
img = (cv2.aruco.generateImageMarker if hasattr(cv2.aruco, 'generateImageMarker')
       else cv2.aruco.drawMarker)(ARUCO_DICT, MARKER_ID, 400)
cv2.imwrite('aruco_marker.png', cv2.copyMakeBorder(img, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255))

def load_stereo():
    s = np.load(STEREO_FILE)
    if str(s['zoom']) != ZOOM.name:
        raise RuntimeError(f"stereo calibration was made at zoom {s['zoom']}, cameras are at {ZOOM.name}")
    return s['K1'], s['D1'], s['K2'], s['D2'], s['R'], s['T'], tuple(int(v) for v in s['size'])

K1, D1, K2, D2, R_lr, T_lr, IMG_SIZE = load_stereo()
R1, R2, P1r, P2r, _, _, _ = cv2.stereoRectify(K1, D1, K2, D2, IMG_SIZE, R_lr, T_lr)
P_LEFT = np.hstack([np.eye(3), np.zeros((3, 1))])   # in normalised (undistorted) coordinates
P_RIGHT = np.hstack([R_lr, T_lr.reshape(3, 1)])

def rectified_y(uv, K, D, R, P):
    return cv2.undistortPoints(np.float32([[uv]]), K, D, R=R, P=P)[0, 0, 1]

def triangulate(uv_left, uv_right):
    # Pixel in each image -> 3D point in the left-camera frame (m) + epipolar error (px).
    nl = cv2.undistortPoints(np.float32([[uv_left]]), K1, D1).reshape(2, 1)
    nr = cv2.undistortPoints(np.float32([[uv_right]]), K2, D2).reshape(2, 1)
    X = cv2.triangulatePoints(P_LEFT, P_RIGHT, nl, nr)
    dy = abs(rectified_y(uv_left, K1, D1, R1, P1r) - rectified_y(uv_right, K2, D2, R2, P2r))
    return (X[:3] / X[3]).ravel(), dy

def detect_marker(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if hasattr(cv2.aruco, 'ArucoDetector'):
        corners, ids, _ = cv2.aruco.ArucoDetector(ARUCO_DICT, cv2.aruco.DetectorParameters()).detectMarkers(gray)
    else:
        corners, ids, _ = cv2.aruco.detectMarkers(gray, ARUCO_DICT, parameters=cv2.aruco.DetectorParameters_create())
    if ids is None or MARKER_ID not in ids.ravel():
        return None
    return corners[list(ids.ravel()).index(MARKER_ID)][0].mean(axis=0)   # marker centre (px)

setup = np.load(SETUP_FILE)
R_GRASP, HEAD_Q = setup['R_grasp'], list(setup['head_q'])

def restore_head():
    stiffen('head', NECK)
    goto({j: a for j, a in zip(NECK, HEAD_Q)}, duration=1.5)

In [ ]:
restore_head()
stiffen(ARM_NAME, CHAIN + [GRIPPER])

# Grid in front of the left arm (y > 0). Tune to your table and reach.
GRID = [np.array([x, y, z]) for x in (0.25, 0.32, 0.39)
                            for y in (0.05, 0.15, 0.25)
                            for z in (-0.30, -0.22, -0.14)]
cam_pts, torso_pts = [], []
for p in GRID:
    q, info = plan_ik(pose(R_GRASP, p), current_q())
    if q is None:
        print(p, 'skip:', info); continue
    move_joints(q, duration=1.5)
    time.sleep(0.7)                                   # let it settle before looking
    left, right = grab_stereo()
    ml, mr = detect_marker(left), detect_marker(right)
    vis = [img.copy() for img in (left, right)]
    for img, m in zip(vis, (ml, mr)):
        if m is not None:
            cv2.circle(img, tuple(int(v) for v in m), 8, (0, 255, 0), 2)
    publish(label(np.hstack(vis), f'grid point {p}  |  {len(cam_pts)} used so far'), hold=1.5)
    if ml is None or mr is None:
        print(p, 'skip: marker not seen in', 'left' if ml is None else 'right'); continue
    X, dy = triangulate(ml, mr)
    if dy > 3:
        print(p, f'skip: epipolar error {dy:.1f} px'); continue
    cam_pts.append(X)
    torso_pts.append(ARM.forward_kinematics()[:3, 3])   # where the arm actually is, not where we asked
    print(p, 'ok')
print(f'{len(cam_pts)} usable points')

In [ ]:
def fit_rigid(A, B):
    # Least-squares R, t with B ~ R @ A + t (Kabsch).
    A, B = np.asarray(A), np.asarray(B)
    ca, cb = A.mean(0), B.mean(0)
    U, _, Vt = np.linalg.svd((A - ca).T @ (B - cb))
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return R, cb - R @ ca

assert len(cam_pts) >= 6, 'need at least 6 points: adjust GRID'
R_CT, t_CT = fit_rigid(cam_pts, torso_pts)
res_mm = np.linalg.norm((np.array(cam_pts) @ R_CT.T + t_CT) - np.array(torso_pts), axis=1) * 1000
print(f'hand-eye residual: mean {res_mm.mean():.1f} mm, max {res_mm.max():.1f} mm  (aim for mean < 10)')
np.savez(HANDEYE_FILE, R=R_CT, t=t_CT, head_q=HEAD_Q)

## Step 4: Find the object in 3D
OWL-ViT runs on both images. The best left detection is paired with the right detection on the same epipolar line,
MobileSAM masks both, and the mask centroids are triangulated. `grasp_debug.jpg` shows what was picked.

The hand–eye fit maps a camera point to *where the arm's end-effector origin must be to put the marker there*.
`GRIP_OFFSET` corrects for the marker not being at the grip centre. With the hand in the taught pose, measure the vector
**from the marker centre to the point between the fingers**, in torso axes (x forward, y left, z up).
`OBJECT_SHIFT` moves the target from the visible surface toward the object's middle, about one radius forward.

In [ ]:
from pollen_vision.vision_models.object_detection import OwlVitWrapper
from pollen_vision.vision_models.object_segmentation import MobileSamWrapper
from pollen_vision.utils import Annotator

owl, sam, annotator = OwlVitWrapper(), MobileSamWrapper(), Annotator()
he = np.load(HANDEYE_FILE)
R_CT, t_CT = he['R'], he['t']

GRIP_OFFSET = np.array([0.0, 0.0, 0.0])    # m, marker centre -> grip centre (torso axes)
OBJECT_SHIFT = np.array([0.02, 0.0, 0.0])  # m, surface centroid -> object middle
DET_THRESHOLD = 0.15
MAX_EPI_PX = 10

def mask_centroid(mask):
    m = cv2.moments(mask.astype(np.uint8))
    return None if m['m00'] == 0 else np.array([m['m10'] / m['m00'], m['m01'] / m['m00']])

def locate(label):
    # Returns the arm end-effector target (torso frame, m) that puts the grip centre on the object, or None.
    left, right = grab_stereo()
    rgb_l, rgb_r = cv2.cvtColor(left, cv2.COLOR_BGR2RGB), cv2.cvtColor(right, cv2.COLOR_BGR2RGB)
    det_l = owl.infer(rgb_l, [label], detection_threshold=DET_THRESHOLD)
    det_r = owl.infer(rgb_r, [label], detection_threshold=DET_THRESHOLD)
    if not det_l or not det_r:
        print(f'"{label}" not seen in', 'left' if not det_l else 'right'); return None
    best_l = max(det_l, key=lambda d: d['score'])
    box_c = lambda d: np.array([(d['box']['xmin'] + d['box']['xmax']) / 2, (d['box']['ymin'] + d['box']['ymax']) / 2])
    y_l = rectified_y(box_c(best_l), K1, D1, R1, P1r)
    best_r = min(det_r, key=lambda d: abs(rectified_y(box_c(d), K2, D2, R2, P2r) - y_l))

    boxes = lambda d: [[d['box'][k] for k in ('xmin', 'ymin', 'xmax', 'ymax')]]
    mask_l = sam.infer(rgb_l, bboxes=boxes(best_l))[0]
    mask_r = sam.infer(rgb_r, bboxes=boxes(best_r))[0]
    cl, cr = mask_centroid(mask_l), mask_centroid(mask_r)
    debug = np.hstack([annotator.annotate(left, [best_l], masks=[mask_l]),
                       annotator.annotate(right, [best_r], masks=[mask_r])])
    cv2.imwrite('grasp_debug.jpg', debug)
    publish(debug, hold=10)
    if cl is None or cr is None:
        print('empty mask'); return None

    X_cam, dy = triangulate(cl, cr)
    if dy > MAX_EPI_PX:
        print(f'left/right detections do not match (epipolar error {dy:.0f} px): see grasp_debug.jpg'); return None
    obj = R_CT @ X_cam + t_CT                 # where the arm origin would put the *marker*
    target = obj + OBJECT_SHIFT - GRIP_OFFSET
    print(f'{label}: score {best_l["score"]:.2f}, camera depth {X_cam[2]:.2f} m, arm target {target}')
    return target

In [ ]:
restore_head()
target = locate('paper cup')

## Step 5: Grasp
Every waypoint is solved with IK **before** anything moves. If one is unreachable or off by more than 1 cm, nothing moves.

In [ ]:
APPROACH = 0.10   # m, pre-grasp distance behind the object (along -x)
LIFT = 0.10       # m

def grasp(target):
    waypoints = [('pre-grasp', target - [APPROACH, 0, 0]),
                 ('grasp', target),
                 ('lift', target + [0, 0, LIFT])]
    plan, q = [], current_q()
    for name, xyz in waypoints:
        q, info = plan_ik(pose(R_GRASP, xyz), q)
        if q is None:
            print(f'{name} {xyz}: {info}. Not moving.'); return False
        print(f'{name:9s} {xyz}  IK error {info:.1f} mm')
        plan.append(q)

    stiffen(ARM_NAME, CHAIN + [GRIPPER])
    set_gripper(GRIPPER_OPEN)
    move_joints(plan[0], 2.5)
    move_joints(plan[1], 1.5)
    set_gripper(GRIPPER_CLOSED, 1.0)
    time.sleep(0.3)
    move_joints(plan[2], 1.5)
    reached = ARM.forward_kinematics()[:3, 3]
    print('done; hand at', reached, '| gripper at', round(GRIPPER.present_position, 1),
          'deg (well above CLOSED means something is in it)')
    return True

if target is not None:
    grasp(target)

In [ ]:
# Put it back down and release
set_gripper(GRIPPER_OPEN)

In [ ]:
# Done: let the arm and head down gently
reachy.turn_off_smoothly(ARM_NAME)
reachy.turn_off_smoothly('head')